In [3]:
# install datasets
!pip install datasets --quiet

# imports
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
from datasets import load_dataset

# load Trelis protein stability dataset
dataset = load_dataset("Trelis/protein_stability_single_mutation")

# convert splits to pandas
df_train = dataset['train'].to_pandas()
df_val = dataset['validation'].to_pandas()
df_test = dataset['test'].to_pandas()

# combine splits
df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)

# small subset
df_sample = df_all.sample(n=5000, random_state=42).reset_index(drop=True)

# save CSV
os.makedirs("../data", exist_ok=True)
df_sample.to_csv("../data/protein_stability_single_mutation.csv", index=False)

print("Dataset subset shape:", df_sample.shape)

In [4]:
# target
y = df_sample['deltaG']

# simulate embeddings
np.random.seed(42)
X = pd.DataFrame(np.random.rand(len(df_sample), 20), columns=[f"feature_{i}" for i in range(20)])

# split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
# Random Forest
rf_model = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_mse = mean_squared_error(y_test, rf_pred)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest Regressor")
print("RMSE:", rf_rmse)
print("R2:", rf_r2)

Random Forest Regressor
RMSE: 1.2810922105529834
R2: -0.007879849151466


In [6]:
# Gradient Boosting
gb_model = GradientBoostingRegressor(n_estimators=50, max_depth=5, random_state=42)
gb_model.fit(X_train, y_train)

gb_pred = gb_model.predict(X_test)

gb_mse = mean_squared_error(y_test, gb_pred)
gb_rmse = np.sqrt(gb_mse)
gb_r2 = r2_score(y_test, gb_pred)

print("Gradient Boosting Regressor")
print("RMSE:", gb_rmse)
print("R2:", gb_r2)

Gradient Boosting Regressor
RMSE: 1.2958210146269609
R2: -0.031188399331372407


In [7]:
# results folder
os.makedirs("../results", exist_ok=True)

# save metrics
metrics = pd.DataFrame({
    "Model": ["RandomForestRegressor", "GradientBoostingRegressor"],
    "RMSE": [rf_rmse, gb_rmse],
    "R2": [rf_r2, gb_r2]
})

metrics_path = "../results/regression_metrics.csv"
metrics.to_csv(metrics_path, index=False)
metrics


,Model,RMSE,R2
0,RandomForestRegressor,1.281092,-0.007880
1,GradientBoostingRegressor,1.295821,-0.031188


In [ ]:
# figures folder
os.makedirs("../figures", exist_ok=True)

plt.figure(figsize=(6,6))
plt.scatter(y_test, rf_pred, alpha=0.6)
plt.xlabel("Actual Stability (ΔΔG)")
plt.ylabel("Predicted Stability")
plt.title("Random Forest: Predicted vs Actual Mutation Stability")
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')  # diagonal line

fig_path = "../figures/regression_results.png"
plt.savefig(fig_path, dpi=300)
plt.show()

print("Figure saved at:", fig_path)

In [11]:
from google.colab import files

metrics.to_csv("regression_metrics.csv", index=False)
files.download("regression_metrics.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>